In [101]:
import pandas
from pyspark.sql.functions import * 
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

In [102]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Read_From_MinIO") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "12345678") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()





In [103]:
df_crm_cust = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/crm/cust/dt=20260430/")

df_crm_prd = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/crm/prd/dt=20260430/")


df_crm_sales = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/crm/sales/dt=20260430/")

df_erp_cat = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/erp/cat/dt=20260430/")

df_erp_cust = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/erp/cust/dt=20260430/")
    
df_erp_loc = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/erp/loc/dt=20260430/")



silver_erp_cust
silver_crm_cust
silver_erp_loc 
--> dim_customers

In [104]:
df_erp_cust.show(3)
print ('crm cust')
df_crm_cust.show(3)
print ('erp_loc')
df_erp_loc.show(3)

+----------+----------+----+----------+
|       CID|     BDATE| GEN| load_date|
+----------+----------+----+----------+
|AW00011000|1971-10-06|MALE|2026-05-01|
|AW00011001|1976-05-10|MALE|2026-05-01|
|AW00011002|1971-02-09|MALE|2026-05-01|
+----------+----------+----+----------+
only showing top 3 rows

crm cust
+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |           Married|    Male|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |            Single|    Male|     2025-10-06|2026-05-01|
| 11002|AW00011002|        Ruben|      Torres|           Married|    Male|     2025-10-06|2026-05-01|
+------+----------+-------------+------------+------------------+--------+

In [105]:
df_erp_cust.show(1)
df_crm_cust.show(1)
df_erp_loc.show(1)

+----------+----------+----+----------+
|       CID|     BDATE| GEN| load_date|
+----------+----------+----+----------+
|AW00011000|1971-10-06|MALE|2026-05-01|
+----------+----------+----+----------+
only showing top 1 row

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |           Married|    Male|     2025-10-06|2026-05-01|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
only showing top 1 row

+----------+---------+----------+
|       CID|    CNTRY| load_date|
+----------+---------+----------+
|AW00011000|AUSTRALIA|2026-05-01|
+----------+---------+----------+
only showing top 1 row



In [106]:
df_crm_cust.select(col('cst_gndr')).distinct().show()

+--------+
|cst_gndr|
+--------+
|  Female|
|     N/a|
|    Male|
+--------+



In [107]:
df_dim_customers = df_crm_cust.alias("crm_c").join(df_erp_cust.alias("erp_c"), col("crm_c.cst_key") == col("erp_c.CID") , "left")\
            .join(df_erp_loc.alias("erp_l"), col("crm_c.cst_key") == col("erp_l.CID"), "left")\
            .select(
                col("crm_c.cst_id"),\
                col("crm_c.cst_key"),\
                col("crm_c.cst_firstname"),\
                col("crm_c.cst_lastname"),\
                col("erp_l.CNTRY"),\
                col("crm_c.cst_marital_status"),\
                # when (col("crm_c.cst_gndr") != "N/a", col("crm_c.cst_gndr")).otherwise(col("erp.GEN"))
                when(
                    col("crm_c.cst_gndr") != "N/a",
                    col("crm_c.cst_gndr")
                ).otherwise(col("erp_c.GEN")).alias("cst_gndr"),\
                col("erp_c.bdate"),\
                col("crm_c.cst_create_date"),\
            )


# select(
#     col("a.id"),
#     col("a.name"),
#     col("b.address")
# )
# CREATE VIEW gold.dim_customers AS
# SELECT
#     ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key, -- Surrogate key
#     ci.cst_id                          AS customer_id,
#     ci.cst_key                         AS customer_number,
#     ci.cst_firstname                   AS first_name,
#     ci.cst_lastname                    AS last_name,
#     la.cntry                           AS country,
#     ci.cst_marital_status              AS marital_status,
#     CASE 
#         WHEN ci.cst_gndr != 'n/a' THEN ci.cst_gndr -- CRM is the primary source for gender
#         ELSE COALESCE(ca.gen, 'n/a')  			   -- Fallback to ERP data
#     END                                AS gender,
#     ca.bdate                           AS birthdate,
#     ci.cst_create_date                 AS create_date
# FROM silver.crm_cust_info ci
# LEFT JOIN silver.erp_cust_az12 ca
#     ON ci.cst_key = ca.cid
# LEFT JOIN silver.erp_loc_a101 la
#     ON ci.cst_key = la.cid;
# GO

silver_crm_prd
silver erp_cat
--> dim_product


In [108]:
# CREATE VIEW gold.dim_products AS
# SELECT
#     ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key, -- Surrogate key
#     pn.prd_id       AS product_id,
#     pn.prd_key      AS product_number,
#     pn.prd_nm       AS product_name,
#     pn.cat_id       AS category_id,
#     pc.cat          AS category,
#     pc.subcat       AS subcategory,
#     pc.maintenance  AS maintenance,
#     pn.prd_cost     AS cost,
#     pn.prd_line     AS product_line,
#     pn.prd_start_dt AS start_date
# FROM silver.crm_prd_info pn
# LEFT JOIN silver.erp_px_cat_g1v2 pc
#     ON pn.cat_id = pc.id
# WHERE pn.prd_end_dt IS NULL; -- Filter out all historical data
# GO

In [109]:
df_crm_prd.show(1)
df_erp_cat.show(2)

+------+-------+--------------------+--------+--------+------------+----------+----------+
|prd_id|prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt| load_date|
+------+-------+--------------------+--------+--------+------------+----------+----------+
|   478|  AC_BC|Mountain Bottle Cage|       4|Mountain|  2013-07-01|      NULL|2026-05-01|
+------+-------+--------------------+--------+--------+------------+----------+----------+
only showing top 1 row

+-----+-----------+-----------+-----------+----------+
|   ID|        CAT|     SUBCAT|MAINTENANCE| load_date|
+-----+-----------+-----------+-----------+----------+
|AC_BR|ACCESSORIES| Bike Racks|        YES|2026-05-01|
|AC_BS|ACCESSORIES|Bike Stands|         NO|2026-05-01|
+-----+-----------+-----------+-----------+----------+
only showing top 2 rows



In [110]:

df_dim_products = df_crm_prd.alias("crm_p").join(df_erp_cat.alias("erp_ca"),\
                        col("crm_p.prd_key") == col("erp_ca.ID"), "left")\
                        .select(\
                            col("crm_p.prd_id"),\
                          row_number().over(
                                Window.orderBy(
                                    col("crm_p.prd_start_dt"),
                                    col("crm_p.prd_key")
                                )
                            ).alias("prd_key"),
                            col("crm_p.prd_nm"),\
                            col("erp_ca.ID"),\
                            col("erp_ca.SUBCAT"),\
                            col("erp_ca.MAINTENANCE"),\
                            col("crm_p.prd_cost"),\
                            col("crm_p.prd_line"),\
                            col("crm_p.prd_start_dt")\
                        ).filter(col("crm_p.prd_end_dt").isNull())

In [111]:
# CREATE VIEW gold.dim_products AS
# SELECT
#     ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key, -- Surrogate key
#     pn.prd_id       AS product_id,
#     pn.prd_key      AS product_number,
#     pn.prd_nm       AS product_name,
#     pn.cat_id       AS category_id,
#     pc.cat          AS category,
#     pc.subcat       AS subcategory,
#     pc.maintenance  AS maintenance,
#     pn.prd_cost     AS cost,
#     pn.prd_line     AS product_line,
#     pn.prd_start_dt AS start_date
# FROM silver.crm_prd_info pn
# LEFT JOIN silver.erp_px_cat_g1v2 pc
#     ON pn.cat_id = pc.id
# WHERE pn.prd_end_dt IS NULL; -- Filter out all historical data
# GO

crm_sales
dim_products
dim_customers 
--> fact_sales

In [112]:
CREATE VIEW gold.fact_sales AS
SELECT
    sd.sls_ord_num  AS order_number,
    pr.product_key  AS product_key,
    cu.customer_key AS customer_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt  AS shipping_date,
    sd.sls_due_dt   AS due_date,
    sd.sls_sales    AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price    AS price
FROM silver.crm_sales_details sd
LEFT JOIN gold.dim_products pr
    ON sd.sls_prd_key = pr.product_number
LEFT JOIN gold.dim_customers cu
    ON sd.sls_cust_id = cu.customer_id;
GO

SyntaxError: invalid syntax (3972537.py, line 1)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price| load_date|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|   3578.0|         1.0|   3578.0|2026-05-01|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|   3400.0|         1.0|   3400.0|2026-05-01|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  20110110|   3400.0|         1.0|   3400.0|2026-05-01|
|    SO43700| BK-R50B-62|      14501|    20101229|   20110105|  20110110|    699.0|         1.0|    699.0|2026-05-01|
|    SO43701| BK-M82S-44|      11003|    20101229|   20110105|  20110110|   3400.0|         1.0|   3400.0|2026-05-01|
+-----------+-----------+-----------+------------+------

In [128]:
df_crm_sales.count()


60398

In [129]:
df_dim_products.count()


295

In [127]:
df_dim_customers.count()

18484

In [140]:
df_crm_sales.show(1)
df_dim_customers.show(1)
df_dim_products.show(1)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price| load_date|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|   3578.0|         1.0|   3578.0|2026-05-01|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
only showing top 1 row

+------+----------+-------------+------------+---------+------------------+--------+----------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|    CNTRY|cst_marital_status|cst_gndr|     bdate|cst_create_date|
+------+----------+-------------+------------+---------+------------------+--------+----------+---------------+
| 11000|AW00011000|          Jon|       Yang |AUST

26/05/04 12:30:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 12:30:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 12:30:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 12:30:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 12:30:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 12:30:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

In [144]:
df_gold_fact_table = df_crm_sales.alias("crm_s").join (df_dim_products.alias("dim_p"), col("crm_s.sls_prd_key") == col("dim_p.prd_key") , "left")\
                            .join (df_dim_customers.alias("dim_c"), col("crm_s.sls_cust_id") == col ("dim_c.cst_id"), "left")\
                            .select (\
                                col("crm_s.sls_ord_num"),\
                                col("dim_p.prd_key"),\
                                col("dim_c.cst_key"),\
                                col("crm_s.sls_order_dt"),\
                                col("crm_s.sls_ship_dt"),\
                                col("crm_s.sls_due_dt"),\
                                col("crm_s.sls_sales"),\
                                col("crm_s.sls_quantity"),\
                                col("crm_s.sls_price")\
                            )